# <center> Elementy numerycznej algebry liniowej </center>

Rozwiązywanie układów równań liniowych jest jednym z podstawowych problemów metod numerycznych. Układy równań liniowych występują w wielu dziedzinach nauki i inżynierii. Stosuje się też w uczeniu maszynowym np. podczas regresji z błędem średniokwadratowym. 


Istnieje kilka metod rozwiązywania układów równań. Na dzisiejszych zajęciach zajmiemy się:
* eliminacją Gaussa bez oraz z wyborem elementu głównego,
* metodami iteracyjnymi.

Problem rozwiązywania układu równań liniowych będzie nam towarzyszły do końca zajęć z tego przedmiotu.

## Normy i wskaźniki uwarunkowania

Wrażliwość układu (zmiana rozwiązania) na niewielkie zaburzenia wektora `b` zależy od macierzy `A` i ocenia się ja za pomocą tzw. współczynnika lub [wskaźnika uwarunkowania macierzy](https://pl.wikipedia.org/wiki/Wskaźnik_uwarunkowania) (ang. *condition number*). Im wyższa wartość tego wskaźnika. tym macierz jest gorzej uwarunkowana. Wskaźnik uwarunkowania to iloczyn normy macierzy z normą jej odwrotności.

$$cond(A)=|A|_{p}\cdot|A^{-1}|_{p}$$
gdzie *p* oznacza jedną z norm macierzy.

In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

***Zadanie 1.***

Porównaj normy 1,2, $\infty$ następujących macierzy:
* [Hilberta](https://pl.wikipedia.org/wiki/Macierz_Hilberta): o wymiarach 5x5 i 15x15
* [Vandermonde'a](https://pl.wikipedia.org/wiki/Macierz_Vandermonde’a): o wymiarach 5x5 i 15x15
* losowej o wartościach z przedziału [0,1]:  o wymiarach 5x5 i 15x15
* $P=\left[\begin{array}{cccc}4 & 1 & -1 & 0 \\ 1 & 3 & -1 & 0 \\ -1 & -1 & 5 & 2 \\ 0 & 0 & 2 & 4\end{array}\right]$

Czy wśród powyższych macierzy jest macierz [diagonalnie dominująca](https://pl.wikipedia.org/wiki/Macierz_przekątniowo_dominująca)?


In [1]:
import numpy as np
from scipy.linalg import hilbert
import pandas as pd

def check_diag_dominant(A):
    abs_diag = np.abs(np.diag(A))
    row_sum = np.sum(np.abs(A), axis=1) - abs_diag
    return np.all(abs_diag > row_sum)

# 1. Definicja macierzy P
P = np.array([
    [4, 1, -1, 0],
    [1, 3, -1, 0],
    [-1, -1, 5, 2],
    [0, 0, 2, 4]
])

# 2. Lista macierzy do przetestowania
test_cases = [
    ("Hilbert 5x5", hilbert(5)),
    ("Hilbert 15x15", hilbert(15)),
    ("Vandermonde 5x5", np.vander(np.linspace(0.1, 1.0, 5))),
    ("Vandermonde 15x15", np.vander(np.linspace(0.1, 1.0, 15))),
    ("Random 5x5", np.random.rand(5, 5)),
    ("Random 15x15", np.random.rand(15, 15)),
    ("Macierz P", P)
]

results = []

for name, A in test_cases:
    # Obliczanie norm
    n1 = np.linalg.norm(A, ord=1)
    n2 = np.linalg.norm(A, ord=2)
    ninf = np.linalg.norm(A, ord=np.inf)
    
    # Obliczanie wskaźnika uwarunkowania (norma 2)
    # cond(A) = ||A|| * ||A^-1||
    cond = np.linalg.cond(A, p=2)
    
    # Sprawdzanie dominacji diagonalnej
    dd = check_diag_dominant(A)
    
    results.append([name, n1, n2, ninf, cond, dd])

# Prezentacja w tabeli
df = pd.DataFrame(results, columns=["Macierz", "Norma 1", "Norma 2", "Norma inf", "cond(A)", "Diag. Domin."])
pd.options.display.float_format = '{:.2e}'.format
print(df.to_string(index=False))

          Macierz  Norma 1  Norma 2  Norma inf  cond(A)  Diag. Domin.
      Hilbert 5x5 2.28e+00 1.57e+00   2.28e+00 4.77e+05         False
    Hilbert 15x15 3.32e+00 1.85e+00   3.32e+00 7.36e+17         False
  Vandermonde 5x5 5.00e+00 3.06e+00   5.00e+00 1.22e+03         False
Vandermonde 15x15 1.50e+01 5.87e+00   1.50e+01 3.05e+12         False
       Random 5x5 3.15e+00 2.54e+00   2.83e+00 1.84e+01         False
     Random 15x15 1.03e+01 7.67e+00   9.39e+00 1.18e+02         False
        Macierz P 9.00e+00 7.09e+00   9.00e+00 3.54e+00          True


*Wskazówka: Do wyznaczenia norm możesz wykorzystać funkcję `numpy.linalg.norm`*

***Zadanie 2.***

Oblicz wskaźniki uwarunkowania macierzy z poprzedniego zadania.

*Wskazówka: Możesz wykorzystać funkcję `numpy.linalg.cond`.*

In [2]:
# zrobione wyżej
import numpy as np
from scipy.linalg import hilbert

# Funkcja pomocnicza do wyświetlania wyników
def print_cond(name, A):
    # cond = ||A|| * ||A^-1||
    c1 = np.linalg.cond(A, p=1)
    c2 = np.linalg.cond(A, p=2)
    cinf = np.linalg.cond(A, p=np.inf)
    print(f"{name:20} | cond_1: {c1:.2e} | cond_2: {c2:.2e} | cond_inf: {cinf:.2e}")

# Definicje macierzy z Zadania 1
P = np.array([[4, 1, -1, 0], [1, 3, -1, 0], [-1, -1, 5, 2], [0, 0, 2, 4]])
nodes_5 = np.linspace(0.1, 1.0, 5)
nodes_15 = np.linspace(0.1, 1.0, 15)

print(f"{'Macierz':20} | {'Wskaźniki uwarunkowania (cond)':^45}")
print("-" * 80)

print_cond("Macierz P", P)
print_cond("Hilbert 5x5", hilbert(5))
print_cond("Hilbert 15x15", hilbert(15))
print_cond("Vandermonde 5x5", np.vander(nodes_5))
print_cond("Vandermonde 15x15", np.vander(nodes_15))

Macierz              |        Wskaźniki uwarunkowania (cond)        
--------------------------------------------------------------------------------
Macierz P            | cond_1: 5.19e+00 | cond_2: 3.54e+00 | cond_inf: 5.19e+00
Hilbert 5x5          | cond_1: 9.44e+05 | cond_2: 4.77e+05 | cond_inf: 9.44e+05
Hilbert 15x15        | cond_1: 7.74e+17 | cond_2: 7.36e+17 | cond_inf: 8.03e+17
Vandermonde 5x5      | cond_1: 2.52e+03 | cond_2: 1.22e+03 | cond_inf: 2.86e+03
Vandermonde 15x15    | cond_1: 1.03e+13 | cond_2: 3.05e+12 | cond_inf: 1.11e+13


## Rozwiązywanie układów równań metodą eliminacji Gaussa

***Zadanie 3.***

Jedną z metod rozwiązywania układów równań liniowych jest metoda eliminacji Gaussa. Metoda ta występuje w kilku odmianach. Poza podstawowym wariantem, możliwe jest zastosowanie metody z wyborem elementu głownego (tzw. *pivoting*). 

Celem tego zadania jest porównanie błędów rozwiązania otrzymanego z tych dwóch wariantów eliminacji Gaussa. Poniżej znajdują się implementacje obu tych metod. Każda z funkcji przyjmuje macierz `A` oraz wektor prawej strony równania `b`.

Samo polecenie znajduje się poniżej.

In [3]:
def gauss_pivot(A, b):
    A=A.copy()
    b=b.copy()
    n = len(b)
    for k in range(n-1):
        ind_max = k
        for j in range(k+1, n):
            if abs(A[j,k]) > abs(A[ind_max,k]):
                ind_max = j
        if ind_max > k:
            tmp = A[ind_max,k:n].copy()
            A[ind_max,k:n] = A[k,k:n]
            A[k,k:n] = tmp
            tmp = b[ind_max].copy()
            b[ind_max] = b[k]
            b[k] = tmp
        akk = A[k,k]
        l = A[k+1:n,k] / akk
        for i in range(k+1, n):
            A[i,k] = 0
            A[i,k+1:n] = A[i,k+1:n] - l[i-k-1] * A[k,k+1:n]
            b[i] = b[i] - l[i-k-1] * b[k]
    x = np.zeros(n)
    x[n-1] = b[n-1]/A[n-1,n-1]
    for k in range(n-2, -1, -1):
        x[k] = (b[k] - np.dot(A[k,k+1:n], x[k+1:n])) / A[k,k]
    return x

In [4]:
def gauss(A, b):
    A=A.copy()
    b=b.copy()
    n = len(b)
    for k in range(n-1):
        akk = A[k,k]
        l = A[k+1:n,k] / akk
        for i in range(k+1, n):
            A[i,k] = 0
            A[i,k+1:n] = A[i,k+1:n] - l[i-k-1] * A[k,k+1:n]
            b[i] = b[i] - l[i-k-1] * b[k]
    x = np.zeros(n)
    x[n-1] = b[n-1] / A[n-1,n-1]
    for k in range(n-2, -1, -1):
        x[k] = (b[k] - np.dot(A[k,k+1:n], x[k+1:n])) / A[k,k]
    return x

Stwórz macierze wartości losowych `A` o wymiarach 10x10 oraz wektor `b` o odpowiednich wymiarach. 
Chcemy rozwiązać układ równań `Ax=b` metodami eliminacji Gaussa bez oraz z wyborem elementu głównego, a następnie porównać dokładność wyników. Metoda z wyborem elementu głównego powinna dawać mniejszy błąd w przypadku dużych wartości znajdujących się na przekątnej. Sprawdź czy to prawda powtarzając obliczenia z  macierzami `A` zawierającym na pierwszym elemencie przekątnej coraz to mniejsze wartości (tak aby wzrosło znaczenie dalszych elementów na przękątnej i tym samym uaktywnił się wybór innego niż pierwszy elementu głównego).

Wskazówka:Do porównania możesz wykorzystać residuum. Jeżeli `x` jest rozwiązaniem układu to `Ax` powinno być równe `b`. Residuum to różnica pomiędzy `b` oraz `Ax`: `res=|b-Ax|`. Możesz porównać zawartości poszczególnych elementów lub obliczyć jakąś normę z otrzymanego wektora.

In [5]:
import numpy as np

# 1. Tworzenie macierzy 10x10 i wektora b
n = 10
A_orig = np.random.rand(n, n)
b_orig = np.random.rand(n)

# Celowe osłabienie pierwszego elementu przekątnej (zwiększenie błędu bez pivotingu)
A_weak = A_orig.copy()
A_weak[0, 0] = 1e-15 

# 2. Rozwiązanie układu oboma metodami
x_gauss = gauss(A_weak, b_orig)
x_pivot = gauss_pivot(A_weak, b_orig)

# 3. Obliczenie residuum: res = ||b - Ax||
# Używamy normy euklidesowej (L2)
res_gauss = np.linalg.norm(b_orig - np.dot(A_weak, x_gauss))
res_pivot = np.linalg.norm(b_orig - np.dot(A_weak, x_pivot))

print(f"Residuum (Gauss podstawowy): {res_gauss:.2e}")
print(f"Residuum (Gauss z pivotingiem): {res_pivot:.2e}")

Residuum (Gauss podstawowy): 5.49e+01
Residuum (Gauss z pivotingiem): 5.65e-15


## Metody iteracyjne

Innym sposobem na rozwiązanie układu równań liniowych jest wykorzystanie metod iteracyjnych, które generują ciągi przybliżeń wektora stanowiącego rozwiązanie układu. Państwa zadaniem będzie implementacja i porównanie zbieżności trzech najpopularniejszych metod iteracyjnego rozwiązywania układów równań liniowych

***Zadanie 4.***

Porównanie zbieżności metod Jacobiego, Gaussa-Seidla i Younga (SOR).
* Zaimplementuj solvery rozwiązujące układy równań metodami Jacobiego, Gaussa-Seidela  i Younga (SOR). Każda funkcja powinna przyjmować macierz A i wektor prawej strony b. Dla uproszczenia, dopuszczalne jest wykorzystanie  inv dla obliczenia macierzy odwrotnej do macierzy trójkątnej (w metodzie G-S i Younga).
* Porównaj zbieżność ciągów iteracyjnych otrzymanych 3 metodami dla 3 układów równań (3 macierzy). W metodzie Younga możesz przyjąć np. $ω = 1.2$.
* Dla macierzy, dla której metoda Younga okazała się zbieżna, porównaj zbieżność ciągów iteracyjnych otrzymanych dla wartości $0 < ω < 3$ (dodatkowe).
* Dla jakiej wartości parametru $ω$ zbieżność ciągu iteracyjnego jest najlepsza? Wynik otrzymany na podstawie obserwacji ciągu odchyleń od rozwiązania dokładnego należy porównać z wnioskiem płynącym z wykresu zależności promienia spektralnego macierzy iteracji w zależności od parametru $ω$ (dodatkowe).

In [6]:
import numpy as np

def jacobi(A, b, tol=1e-10, max_iter=1000):
    n = len(b)
    x = np.zeros(n)
    D = np.diag(np.diag(A))
    LU = A - D
    D_inv = np.diag(1 / np.diag(D))
    
    iters = 0
    for k in range(max_iter):
        iters += 1
        x_new = D_inv @ (b - LU @ x)
        if np.linalg.norm(x_new - x) < tol:
            return x_new, iters
        x = x_new
    return x, iters

def gauss_seidel(A, b, tol=1e-10, max_iter=1000):
    n = len(b)
    x = np.zeros(n)
    L = np.tril(A)  # Macierz trójkątna dolna (D + L)
    U = A - L       # Macierz trójkątna górna
    L_inv = np.linalg.inv(L)
    
    iters = 0
    for k in range(max_iter):
        iters += 1
        x_new = L_inv @ (b - U @ x)
        if np.linalg.norm(x_new - x) < tol:
            return x_new, iters
        x = x_new
    return x, iters

def sor_young(A, b, omega, tol=1e-10, max_iter=1000):
    n = len(b)
    x = np.zeros(n)
    D = np.diag(np.diag(A))
    L = np.tril(A, -1)
    U = np.triu(A, 1)
    
    # Macierz M_sor = (D + wL)^-1
    M_inv = np.linalg.inv(D + omega * L)
    
    iters = 0
    for k in range(max_iter):
        iters += 1
        # x^(k+1) = (D + wL)^-1 * [w*b - (wU + (w-1)D) * x^(k)]
        x_new = M_inv @ (omega * b - (omega * U + (omega - 1) * D) @ x)
        if np.linalg.norm(x_new - x) < tol:
            return x_new, iters
        x = x_new
    return x, iters


## Porównanie rozwiązania za pomocą metody `solve` oraz z użyciem odwrotności na przykładzie macierzy źle uwarunkowanej

***Zadanie 5.***

Dany jest układ równań $Hx=b$.
* H jest macierzą Hilberta o wymiarach $n=5x5$ (I przypadek) i $n=15x15$ (II przypadek),
* b jest wektorem o następujących elementach $b_i = 1/(n + i + 1)$ Uwaga: $i=1,\dots,n$.

Do rozwiązania układu wykorzystaj dwa algorytmy:
1. Z odwracaniem macierzy współczynników H.
2. Metodę `numpy.linalg.solve`.

Porównaj błędy obu rozwiązań. Aby ocenić błąd możesz:
* wyznaczyć wektor residuum otrzymanego rozwiązania,
* rozwiązać układ równań z innym wektorem $b$. Załóż, że wektor rozwiązania ma wszystkie elementy (współrzędne) równe 1 ($u_i = 1, i = 1, 2, . . . , n$). Wtedy $b = Hu$. Układ rozwiążemy bez korzystania z wiedzy o postaci $u$. Dopiero wynik porównamy ze znanym nam $u$.

In [7]:
import numpy as np
from scipy.linalg import hilbert

def solve_hilbert_test(n):
    # 1. Tworzenie macierzy Hilberta H
    H = hilbert(n)
    
    # 2. Tworzenie wektora b tak, aby rozwiązaniem był wektor jedynek u
    # Zgodnie ze wskazówką: b = H * u, gdzie u_i = 1
    u_true = np.ones(n)
    b = H @ u_true
    
    # 3. Rozwiązanie metodą numpy.linalg.solve (rekomendowana)
    x_solve = np.linalg.solve(H, b)
    
    # 4. Rozwiązanie przez jawną odwrotność (niezalecana)
    x_inv = np.linalg.inv(H) @ b
    
    # 5. Obliczenie błędów (norma z różnicy rozwiązań)
    err_solve = np.linalg.norm(u_true - x_solve)
    err_inv = np.linalg.norm(u_true - x_inv)
    
    # 6. Obliczenie residuum: ||b - Hx||
    res_solve = np.linalg.norm(b - H @ x_solve)
    res_inv = np.linalg.norm(b - H @ x_inv)
    
    return (err_solve, err_inv, res_solve, res_inv, np.linalg.cond(H))

# Przeprowadzenie testów
for n in [5, 15]:
    e_s, e_i, r_s, r_i, c = solve_hilbert_test(n)
    print(f"Wyniki dla n = {n} (cond = {c:.2e}):")
    print(f"  Metoda solve: Błąd = {e_s:.2e}, Residuum = {r_s:.2e}")
    print(f"  Metoda inv:   Błąd = {e_i:.2e}, Residuum = {r_i:.2e}")
    print("-" * 50)

Wyniki dla n = 5 (cond = 4.77e+05):
  Metoda solve: Błąd = 8.46e-12, Residuum = 2.48e-16
  Metoda inv:   Błąd = 2.94e-11, Residuum = 1.03e-11
--------------------------------------------------
Wyniki dla n = 15 (cond = 7.36e+17):
  Metoda solve: Błąd = 9.80e+00, Residuum = 7.11e-16
  Metoda inv:   Błąd = 1.81e+03, Residuum = 1.99e+00
--------------------------------------------------


**Zadanie domowe. Znaczenie wskaźnika uwarunkowania macierzy w szacowaniu błędu rozwiązania**


Dana jest następująca macierz A współczynników układu dwóch równań liniowy:
$$A=\begin{bmatrix}10^5 & 9.9\cdot10^4\\1.00001& 0.99\end{bmatrix}$$

Wektor prawej strony równania $Ax=b$ dla rozwiązania x = $[1, 1]^T$ możemy wyznaczyć z równości $b = Ax$.

Należy:
* obliczyć wskaźnik uwarunkowania macierzy $A$,
* rozwiązać układ równań $Ax = b$ (nie korzystając z wiedzy o przyjętym rozwiązaniu dokładnym x) korzystając z funkcji `np.linalg.solve`,
* ocenić błąd otrzymanego rozwiązania i porównać go z błędem szacowanym za pomocą wskaźnika uwarunkowania macierzy A,
* przeprowadzić skalowanie tak, aby macierz $A$ była wyważona wierszami,
* wyznaczyć nowe wartości wektora b tak, aby rozwiązanie dokładne się nie
zmieniło,
* obliczyć wskaźnik uwarunkowania macierzy skalowanej,
* rozwiązać układ równań tą samą metodą jak poprzednio,
* ocenić błąd otrzymanego rozwiązania i porównać go z błędem szacowanym za pomocą wskaźnika uwarunkowania skalowanej macierzy $A$.
1. Czy błąd numeryczny rozwiązania w obu przypadkach jest tego samego rzędu?
2. Które szacowanie błędu jest bardziej zbliżone do faktycznego błędu?

In [8]:
import numpy as np

# 1. Definicja macierzy A i wektora b
A = np.array([
    [10**5, 9.9 * 10**4],
    [1.00001, 0.99]
])

# Dokładne rozwiązanie x = [1, 1]^T
x_exact = np.array([1.0, 1.0])
b = A @ x_exact

def analyze_system(A_sys, b_sys, label):
    # Obliczenie wskaźnika uwarunkowania (norma nieskończoność)
    cond_inf = np.linalg.cond(A_sys, p=np.inf)
    
    # Rozwiązanie układu
    x_calc = np.linalg.solve(A_sys, b_sys)
    
    # Obliczenie błędu bezwzględnego i względnego
    error_abs = np.linalg.norm(x_exact - x_calc, ord=np.inf)
    error_rel = error_abs / np.linalg.norm(x_exact, ord=np.inf)
    
    print(f"--- {label} ---")
    print(f"Wskaźnik uwarunkowania (cond_inf): {cond_inf:.2e}")
    print(f"Obliczone x: {x_calc}")
    print(f"Błąd względny: {error_rel:.2e}\n")
    return cond_inf, error_rel

# ANALIZA 1: Układ oryginalny
c1, e1 = analyze_system(A, b, "PRZED SKALOWANIEM")

# ANALIZA 2: Skalowanie (wyważenie wierszami)
# Skalujemy tak, aby największy element w każdym wierszu miał moduł 1
S = np.diag(1.0 / np.max(np.abs(A), axis=1))
A_scaled = S @ A
b_scaled = S @ b # b musi być przeskalowane tym samym wektorem!

c2, e2 = analyze_system(A_scaled, b_scaled, "PO SKALOWANIU")


--- PRZED SKALOWANIEM ---
Wskaźnik uwarunkowania (cond_inf): 2.01e+10
Obliczone x: [1. 1.]
Błąd względny: 1.12e-11

--- PO SKALOWANIU ---
Wskaźnik uwarunkowania (cond_inf): 4.02e+05
Obliczone x: [1. 1.]
Błąd względny: 0.00e+00

